In [ ]:
import argparse
import asyncio
import os
import re
from urllib.parse import urlparse
from urllib.parse import quote
import json
import requests
import traceback

import pandas as pd
from openai_codex import Codex, Sandbox
from openai import OpenAI

from agents import Agent, Runner, set_default_openai_key, ModelSettings
from agents.mcp import MCPServerStreamableHttp

from tools.text_chunker import split_log_into_chunks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import time
from pathlib import Path
from github import Github
import requests
import jsonlines 

In [ ]:
TOKEN = 
OPENAI_API_KEY = 
gemini_key = 

OWNER = "pytorch"
REPO = "pytorch"
BRANCH = "main"

REQUEST_DELAY = 2          # seconds between API requests
SAVE_EVERY = 5           # save progress every N commits
PER_PAGE = 100

OUTPUT_FILE = "pytorch_transition_withlog_0928_part4.csv"
ACTION_FILE = "pytorch_actions_0928.csv"
RAW_LOG_FILE = "pytorch_transition_raw_logs_0928_part4.jsonl"

In [ ]:
gh = Github(TOKEN, per_page=100)
repo = gh.get_repo(f"{OWNER}/{REPO}")

In [ ]:
df_actions = pd.read_csv(ACTION_FILE)
len(df_actions)

In [ ]:
df_actions.columns

In [ ]:
start_time = df_actions['workflow_created_at'].min()
end_time = df_actions['workflow_created_at'].max()

print(f"Start: {start_time}, End: {end_time}")

In [ ]:
##ci_df = df_actions[:30000].copy()
#ci_df = df_actions[30000:80000].copy()
#ci_df = df_actions[80000:130000].copy()
#ci_df = df_actions[130000:173684].copy()
ci_df = df_actions.copy()
len(ci_df)

In [ ]:
ci_df["commit_sha"].nunique()

In [ ]:
ci_df["workflow_event"].value_counts()

In [ ]:
ALLOWED_EVENTS = {"push", "schedule"}
ci_df = ci_df[
    ci_df["workflow_event"].isin(ALLOWED_EVENTS)
].copy()
print(ci_df["workflow_event"].value_counts())
print("Rows after event filter:", len(ci_df))

In [ ]:
start_time = ci_df['workflow_created_at'].min()
end_time = ci_df['workflow_created_at'].max()

print(f"Start: {start_time}, End: {end_time}")

In [ ]:
start_time = ci_df['completed_at'].min()
end_time = ci_df['completed_at'].max()

print(f"Start: {start_time}, End: {end_time}")

In [ ]:
sha_col = "commit_sha" if "commit_sha" in ci_df.columns else "workflow_head_sha"

ci_df[sha_col] = (
    ci_df[sha_col]
    .astype("string")
    .str.lower()
    .str.strip()
)

ci_df["workflow_created_at"] = pd.to_datetime(
    ci_df["workflow_created_at"],
    utc=True,
    errors="coerce",
)

ci_shas = set(ci_df[sha_col].dropna())

ci_df["workflow_created_at"] = pd.to_datetime(
    ci_df["workflow_created_at"],
    utc=True,
    errors="coerce",
)

valid_times = ci_df["workflow_created_at"].dropna()

since = valid_times.min() - pd.Timedelta(days=10)
until = valid_times.max() + pd.Timedelta(days=10)

print("Fetching main commits from", since, "to", until)

commits = repo.get_commits(
    sha="main",
    since=since.to_pydatetime(),
    until=until.to_pydatetime(),
)

rows = []

for commit in commits:
    rows.append(
        {
            "sha": commit.sha.lower(),
            "parent_shas": [p.sha.lower() for p in commit.parents],
            "commit_date": commit.commit.committer.date,
            "message": commit.commit.message.splitlines()[0],
        }
    )

commits_df = pd.DataFrame(rows)

print("Fetched main commits:", len(commits_df))
print("CI SHAs:", len(ci_shas))
print("CI SHAs found in fetched main commits:", commits_df["sha"].isin(ci_shas).sum())

In [ ]:
def parse_parent_shas(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    value = str(value).strip()
    if not value:
        return []

    if value.startswith("["):
        return list(ast.literal_eval(value))

    if "," in value:
        return [x.strip() for x in value.split(",") if x.strip()]

    return [value]


In [ ]:
def build_and_validate_main_timeline(commits_df, tip_sha=None):
    """
    Build a first-parent main ancestry timeline.

    Assumption:
      commits_df contains a continuous slice of commits from main.

    Returns:
      timeline_df ordered oldest -> newest with main_pos.

    Raises:
      ValueError if the dataframe does not form one contiguous first-parent chain.
    """
    df = commits_df.copy()
    df["parent_shas_parsed"] = df["parent_shas"].apply(parse_parent_shas)

    all_shas = set(df["sha"])

    first_parent_by_sha = {}
    for _, row in df.iterrows():
        parents = row["parent_shas_parsed"]
        first_parent_by_sha[row["sha"]] = parents[0] if parents else None

    # Infer newest commit if not given.
    # Newest tip is the commit that no other commit uses as first parent.
    if tip_sha is None:
        referenced_as_first_parent = {
            parent
            for parent in first_parent_by_sha.values()
            if parent is not None
        }

        candidate_tips = all_shas - referenced_as_first_parent

        if len(candidate_tips) != 1:
            raise ValueError(
                f"Expected exactly 1 newest tip, found {len(candidate_tips)}: "
                f"{sorted(candidate_tips)[:10]}"
            )

        tip_sha = next(iter(candidate_tips))

    if tip_sha not in all_shas:
        raise ValueError(f"tip_sha is not in dataframe: {tip_sha}")

    # Walk backward newest -> oldest using first parents.
    chain_newest_to_oldest = []
    seen = set()
    current_sha = tip_sha

    while current_sha is not None:
        if current_sha in seen:
            raise ValueError(f"Cycle detected at {current_sha}")

        if current_sha not in all_shas:
            raise ValueError(
                f"Missing internal parent link: {current_sha} is needed "
                f"but is not in dataframe."
            )

        seen.add(current_sha)
        chain_newest_to_oldest.append(current_sha)

        parent = first_parent_by_sha[current_sha]

        # If parent is outside the dataframe, it must be because current_sha
        # is the oldest commit in this collected slice.
        if parent is not None and parent not in all_shas:
            break

        current_sha = parent

    chain_oldest_to_newest = list(reversed(chain_newest_to_oldest))

    visited = set(chain_oldest_to_newest)

    # This catches missing links, wrong tip, duplicate branch fragments, etc.
    unvisited = all_shas - visited
    if unvisited:
        raise ValueError(
            f"Timeline is not one contiguous first-parent chain. "
            f"{len(unvisited)} commits were not visited. Examples: "
            f"{sorted(unvisited)[:10]}"
        )

    # This checks that only one commit has a first parent outside the dataframe:
    # the oldest commit in your 1000-commit slice.
    outside_parent_count = 0
    outside_parent_examples = []

    for sha, parent in first_parent_by_sha.items():
        if parent is not None and parent not in all_shas:
            outside_parent_count += 1
            outside_parent_examples.append((sha, parent))

    if outside_parent_count > 1:
        raise ValueError(
            f"Expected only the oldest commit to have a parent outside the dataframe, "
            f"but found {outside_parent_count}. Examples: {outside_parent_examples[:5]}"
        )

    timeline = (
        df.set_index("sha")
          .loc[chain_oldest_to_newest]
          .reset_index()
    )

    timeline.insert(0, "main_pos", range(len(timeline)))

    return timeline

In [ ]:
timeline_df = build_and_validate_main_timeline(commits_df)

ci_mapped = ci_df.merge(
    timeline_df[["sha", "main_pos", "commit_date"]],
    left_on=sha_col,
    right_on="sha",
    how="left",
)

print("Mapped rows:", ci_mapped["main_pos"].notna().sum())
print("Unmapped rows:", ci_mapped["main_pos"].isna().sum())

In [ ]:
timeline_df = build_and_validate_main_timeline(commits_df)

timeline_df[["main_pos", "sha", 'parent_shas', "commit_date", "message"]].head()
timeline_df[["main_pos", "sha", 'parent_shas', "commit_date", "message"]].tail()

In [ ]:
len(timeline_df)

In [ ]:
timeline_df.to_csv("pytorch_git_timeline_0928_run2.csv")

In [ ]:
actions_df = ci_df.copy()

In [ ]:
job_identity_cols = [
    "workflow_run_id",
    "job_id",
]

duplicate_action_rows = actions_df[
    actions_df.duplicated(subset=job_identity_cols, keep=False)
].copy()

print("duplicate job-run rows:", len(duplicate_action_rows))

In [ ]:
def map_ci_rows_to_main(actions_df, timeline_df):
    df = actions_df.copy()

    timeline_lookup = (
        timeline_df[["sha", "main_pos"]]
        .drop_duplicates(subset=["sha"])
        .rename(columns={"sha": "workflow_head_sha"})
    )

    df = df.merge(
        timeline_lookup,
        on="workflow_head_sha",
        how="left",
        validate="many_to_one",
    )

    df["is_sha_on_main_timeline"] = df["main_pos"].notna()

    branch = df["workflow_head_branch"].fillna("").astype(str).str.strip()
    df["is_head_branch_main"] = branch.isin(["main", "refs/heads/main"])

    # Strict main-relevant rows:
    # SHA is in main timeline AND workflow reports main branch.
    ci_on_main_strict = df[
        df["is_sha_on_main_timeline"] & df["is_head_branch_main"]
    ].copy()

    # SHA-based main rows:
    # Useful if branch metadata is unreliable or missing.
    ci_on_main_by_sha = df[
        df["is_sha_on_main_timeline"]
    ].copy()

    ci_not_on_main = df[
        ~df["is_sha_on_main_timeline"]
    ].copy()

    ci_on_main_strict["main_pos"] = ci_on_main_strict["main_pos"].astype(int)
    ci_on_main_by_sha["main_pos"] = ci_on_main_by_sha["main_pos"].astype(int)

    return ci_on_main_strict, ci_on_main_by_sha, ci_not_on_main, df

In [ ]:
ci_on_main_strict, ci_on_main_by_sha, ci_not_on_main, actions_mapped = (
    map_ci_rows_to_main(actions_df, timeline_df)
)


print("all CI rows:", len(actions_df))
print("SHA on main timeline:", len(ci_on_main_by_sha))
print("strict main branch + SHA on timeline:", len(ci_on_main_strict))
print("not on main timeline:", len(ci_not_on_main))

In [ ]:
ci_on_main_strict["workflow_event"].value_counts()

In [ ]:
def define_logical_ci_tracks(
    ci_df,
    include_workflow_event=False,
    include_runner_group=False,
):
    """
    Define logical CI tracks.

    Default updated-pipeline behavior:
      track = workflow_name + job_name

    If include_workflow_event=True:
      track = workflow_name + workflow_event + job_name

    If include_runner_group=True:
      runner_group_name is included when available.
    """

    df = ci_df.copy()

    track_cols = ["workflow_name", "job_name"]

    if include_workflow_event:
        track_cols = ["workflow_name", "workflow_event", "job_name"]

    if include_runner_group and "runner_group_name" in df.columns:
        # Only include runner_group_name if it has real values.
        has_runner_group = (
            df["runner_group_name"]
            .fillna("")
            .astype(str)
            .str.strip()
            .ne("")
            .any()
        )

        if has_runner_group:
            track_cols.append("runner_group_name")

    for col in track_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df["track_id"] = df[track_cols].agg(" | ".join, axis=1)

    return df, track_cols

In [ ]:
ci_with_tracks, track_cols = define_logical_ci_tracks(
    ci_on_main_strict,
    include_workflow_event=False,
    include_runner_group=False,
)

print("track columns:", track_cols)
print("number of tracks:", ci_with_tracks["track_id"].nunique())

In [ ]:
len(ci_with_tracks)

In [ ]:
ci_with_tracks["track_id"].value_counts()

In [ ]:
def deduplicate_track_commit_observations(ci_df):
    """
    Keep one final CI observation per logical track + commit.

    Requires:
      track_id
      workflow_head_sha
      workflow_run_attempt
      completed_at or workflow_updated_at
    """

    df = ci_df.copy()

    # Make attempt numeric so sorting works correctly.
    df["workflow_run_attempt_num"] = pd.to_numeric(
        df["workflow_run_attempt"],
        errors="coerce",
    ).fillna(0).astype(int)

    # Parse timestamps for stable latest-row selection.
    for col in ["completed_at", "workflow_updated_at", "started_at"]:
        if col in df.columns:
            df[col + "_dt"] = pd.to_datetime(df[col], errors="coerce", utc=True)

    sort_cols = [
        "track_id",
        "workflow_head_sha",
        "workflow_run_attempt_num",
    ]

    if "completed_at_dt" in df.columns:
        sort_cols.append("completed_at_dt")
    elif "workflow_updated_at_dt" in df.columns:
        sort_cols.append("workflow_updated_at_dt")
    elif "started_at_dt" in df.columns:
        sort_cols.append("started_at_dt")

    df = df.sort_values(sort_cols, ascending=True, na_position="last")

    group_cols = ["track_id", "workflow_head_sha"]

    duplicate_obs = df[df.duplicated(subset=group_cols, keep=False)].copy()

    deduped = (
        df
        .drop_duplicates(subset=group_cols, keep="last")
        .reset_index(drop=True)
    )

    return deduped, duplicate_obs

In [ ]:
ci_deduped, duplicate_track_commit_obs = deduplicate_track_commit_observations(
    ci_with_tracks
)

print("rows before dedup:", len(ci_with_tracks))
print("rows after dedup:", len(ci_deduped))
print("duplicate track+commit observations:", len(duplicate_track_commit_obs))
print("unique duplicated track+commit groups:",
      duplicate_track_commit_obs[["track_id", "workflow_head_sha"]].drop_duplicates().shape[0])

In [ ]:
ci_deduped["track_id"].nunique()

In [ ]:
def order_tracks_by_git_ancestry(ci_deduped):
    """
    Order each logical CI track by Git ancestry.

    Requires:
      track_id
      main_pos

    Returns:
      ordered dataframe with track_order added.
    """

    df = ci_deduped.copy()

    df["main_pos"] = pd.to_numeric(df["main_pos"], errors="coerce")

    if df["main_pos"].isna().any():
        raise ValueError("Some rows have missing main_pos. Filter/map CI rows first.")

    df["main_pos"] = df["main_pos"].astype(int)

    # Optional timestamp tie-breakers for stable ordering.
    for col in ["completed_at", "workflow_updated_at", "started_at"]:
        if col in df.columns:
            df[col + "_dt"] = pd.to_datetime(df[col], errors="coerce", utc=True)

    sort_cols = ["track_id", "main_pos"]

    for col in ["completed_at_dt", "workflow_updated_at_dt", "started_at_dt"]:
        if col in df.columns:
            sort_cols.append(col)

    df = (
        df
        .sort_values(sort_cols, ascending=True, na_position="last")
        .reset_index(drop=True)
    )

    df["track_order"] = df.groupby("track_id").cumcount()

    return df

In [ ]:
ci_ordered = order_tracks_by_git_ancestry(ci_deduped)

ci_ordered[
    [
        "track_id",
        "track_order",
        "main_pos",
        "workflow_head_sha",
        "workflow_event",
        "job_conclusion",
    ]
].head(6)

In [ ]:
ci_ordered.columns

In [ ]:
ordered_track_summary = (
    ci_ordered
    .groupby("track_id")
    .agg(
        observations=("job_id", "count"),
        unique_commits=("workflow_head_sha", "nunique"),
        first_main_pos=("main_pos", "min"),
        last_main_pos=("main_pos", "max"),
        conclusions=("job_conclusion", lambda x: sorted(set(x.dropna()))),
        events=("workflow_event", lambda x: sorted(set(x.dropna()))),
    )
    .reset_index()
    .sort_values(["observations", "unique_commits"], ascending=False)
)

ordered_track_summary.head(6)

In [ ]:
def normalize_conclusion(conclusion):
    if pd.isna(conclusion):
        return None
    return str(conclusion).strip().lower()


def is_success(conclusion):
    return normalize_conclusion(conclusion) == "success"


def is_failure(conclusion):
    return normalize_conclusion(conclusion) == "failure"


def is_no_data(conclusion):
    conclusion = normalize_conclusion(conclusion)

    return conclusion in [
        None,
        "",
        "neutral",
        "skipped",
        "pending",
        "cancelled",
        "timed_out",
        "action_required",
    ]

In [ ]:
def find_transitions_for_one_track(job_df):
    """
    Input:
      one track history ordered oldest -> newest by main_pos

    Pattern:
      success -> failure... -> success

    Returns:
      list of transition dictionaries
    """

    transitions = []

    job_df = job_df.sort_values(["main_pos", "track_order"]).reset_index(drop=True)
    states = job_df.to_dict("records")

    i = 0

    while i < len(states):
        current = states[i]

        if not is_success(current["job_conclusion"]):
            i += 1
            continue

        last_good = current
        j = i + 1

        # Skip no-data entries between pass and fail.
        while j < len(states) and is_no_data(states[j]["job_conclusion"]):
            j += 1

        if j >= len(states):
            break

        if not is_failure(states[j]["job_conclusion"]):
            i += 1
            continue

        first_bad = states[j]

        k = j + 1
        failure_count = 1
        last_bad = first_bad

        # Continue through failure streak, skipping no-data.
        while k < len(states):
            conclusion = states[k]["job_conclusion"]

            if is_no_data(conclusion):
                k += 1
                continue

            if is_failure(conclusion):
                failure_count += 1
                last_bad = states[k]
                k += 1
                continue

            break

        # Need restoration success.
        if k < len(states) and is_success(states[k]["job_conclusion"]):
            fix_commit = states[k]

            transitions.append({
                "track_id": first_bad["track_id"],

                "workflow_name": first_bad.get("workflow_name"),
                "job_name": first_bad.get("job_name"),

                # Keep event metadata from each boundary.
                "last_pass_event": last_good.get("workflow_event"),
                "first_fail_event": first_bad.get("workflow_event"),
                "last_fail_event": last_bad.get("workflow_event"),
                "fix_event": fix_commit.get("workflow_event"),

                # Boundary commits.
                "last_pass_sha": last_good["workflow_head_sha"],
                "first_fail_sha": first_bad["workflow_head_sha"],
                "last_fail_sha": last_bad["workflow_head_sha"],
                "fix_sha": fix_commit["workflow_head_sha"],

                # Boundary positions in main ancestry.
                "last_pass_pos": int(last_good["main_pos"]),
                "first_fail_pos": int(first_bad["main_pos"]),
                "last_fail_pos": int(last_bad["main_pos"]),
                "fix_pos": int(fix_commit["main_pos"]),

                # Dates/times.
                "last_pass_commit_date": last_good.get("commit_date"),
                "first_fail_commit_date": first_bad.get("commit_date"),
                "last_fail_commit_date": last_bad.get("commit_date"),
                "fix_commit_date": fix_commit.get("commit_date"),

                "first_fail_completed_at": first_bad.get("completed_at"),
                "last_fail_completed_at": last_bad.get("completed_at"),
                "fix_completed_at": fix_commit.get("completed_at"),

                # Evidence pointers from first failing observation.
                "first_fail_workflow_run_id": first_bad.get("workflow_run_id"),
                "first_fail_job_id": first_bad.get("job_id"),
                "first_fail_workflow_url": first_bad.get("workflow_url"),
                "first_fail_job_url": first_bad.get("job_url"),

                # Evidence pointers from last failing observation.
                "last_fail_workflow_run_id": last_bad.get("workflow_run_id"),
                "last_fail_job_id": last_bad.get("job_id"),
                "last_fail_workflow_url": last_bad.get("workflow_url"),
                "last_fail_job_url": last_bad.get("job_url"),

                # Evidence pointers from fix observation.
                "fix_workflow_run_id": fix_commit.get("workflow_run_id"),
                "fix_job_id": fix_commit.get("job_id"),
                "fix_workflow_url": fix_commit.get("workflow_url"),
                "fix_job_url": fix_commit.get("job_url"),

                # Counts.
                "failure_count": failure_count,
                "observed_failure_span": int(last_bad["main_pos"]) - int(first_bad["main_pos"]) + 1,
            })

            # Continue searching from the restoration success.
            i = k

        else:
            i += 1

    return transitions

In [ ]:
def find_transitions_all_tracks(ci_ordered):
    all_transitions = []

    for track_id, track_df in ci_ordered.groupby("track_id", sort=False):
        track_transitions = find_transitions_for_one_track(track_df)
        all_transitions.extend(track_transitions)

    transitions_df = pd.DataFrame(all_transitions)

    if not transitions_df.empty:
        transitions_df = transitions_df.sort_values(
            ["track_id", "first_fail_pos", "fix_pos"]
        ).reset_index(drop=True)

    return transitions_df

In [ ]:
transitions_df = find_transitions_all_tracks(ci_ordered)

print("transitions:", len(transitions_df))

transitions_df[
    [
        "track_id",
        "last_pass_sha",
        "first_fail_sha",
        "last_fail_sha",
        "fix_sha",
        "last_pass_pos",
        "first_fail_pos",
        "last_fail_pos",
        "fix_pos",
        "failure_count",
    ]
].head(10)

In [ ]:
transitions_df["track_id"].nunique

In [ ]:
len(transitions_df)

In [ ]:
transition_key = [
    "track_id",
    "last_fail_sha",
    "fix_sha",
    "last_fail_workflow_run_id",
    "last_fail_job_id",
]

print("Transitions before dedup:", len(transitions_df))
print(
    "Duplicate transition rows:",
    transitions_df.duplicated(subset=transition_key, keep=False).sum(),
)

In [ ]:
transitions_df["failure_window_commit_count"] = (
    transitions_df["first_fail_pos"] - transitions_df["last_pass_pos"]
)

transitions_df["recovery_window_commit_count"] = (
    transitions_df["fix_pos"] - transitions_df["last_fail_pos"]
)

transitions_df["failure_period_span"] = (
    transitions_df["last_fail_pos"] - transitions_df["first_fail_pos"] + 1
)

transitions_df.head()

In [ ]:
transitions_df.columns

In [ ]:
transitions_df.to_csv("track_based_transitions_enriched_0928_run2.csv")

In [ ]:
transitions_df = pd.read_csv("track_based_transitions_enriched_0928_run2.csv")

In [ ]:
len(transitions_df[transitions_df["failure_count"] == 1])

In [ ]:
len(transitions_df[transitions_df["failure_count"] > 1])

In [ ]:
###log error analysis goes below

In [ ]:
def extract_key_log_lines(log_text: str) -> str:
    if pd.isna(log_text) or not str(log_text).strip():
        return ""
    lines = str(log_text).splitlines()
    keep_patterns = [
        r"##\[error\]",
        r"error:",
    ]
    kept = [
        line for line in lines
        if any(re.search(pat, line, flags=re.IGNORECASE) for pat in keep_patterns)
    ]
    return "\n".join(kept)

In [ ]:
def download_job_logs(logs_url: str) -> str:
    response = requests.get(logs_url, allow_redirects=True, timeout=120)
    response.raise_for_status()
    return response.text

In [ ]:
from datetime import datetime, timezone
LOG_RETENTION_DAYS = 90
RETENTION_SAFETY_DAYS = 1  # avoid edge cases near exactly 90 days

def parse_utc_timestamp(value):
    if value is None or pd.isna(value):
        return None

    ts = pd.to_datetime(value, utc=True, errors="coerce")

    if pd.isna(ts):
        return None

    return ts.to_pydatetime()
    
def is_within_log_retention(
    completed_at=None,
    workflow_created_at=None,
    retention_days=LOG_RETENTION_DAYS,
    safety_days=RETENTION_SAFETY_DAYS,
):
    """
    Return (ok_to_fetch, reason, age_days).

    Prefer completed_at because logs are generated during/after the job.
    Fall back to workflow_created_at if completed_at is unavailable.
    """
    reference_time = parse_utc_timestamp(completed_at)

    if reference_time is None:
        reference_time = parse_utc_timestamp(workflow_created_at)

    if reference_time is None:
        return False, "missing_or_invalid_timestamp", None

    now = datetime.now(timezone.utc)
    age_days = (now - reference_time).total_seconds() / 86400

    max_age = retention_days - safety_days

    if age_days > max_age:
        return False, "outside_log_retention_window", age_days

    return True, "within_log_retention_window", age_days

In [ ]:
def fetch_job_log(
    job,
    job_id,
    workflow_run_id: int,
    last_fail_sha: str,
    raw_log_file: str,
    completed_at=None,
    workflow_created_at=None,
) -> tuple[str | None, str | None, str]:
    """
    Downloads the raw log for `job`, appends it to `raw_log_file` JSONL,
    and returns (filtered_log_text, logs_url, fetch_status).
    """
    ok_to_fetch, retention_reason, age_days = is_within_log_retention(
        completed_at=completed_at,
        workflow_created_at=workflow_created_at,
    )
    print(
        f"  retention_check: {retention_reason}, "
        f"completed_at={completed_at}, age_days={age_days}"
    )

    if not ok_to_fetch:
        return None, None, retention_reason

    try:
        logs_url = job.logs_url()
        raw_log = download_job_logs(logs_url)
        filtered = extract_key_log_lines(raw_log)

        with jsonlines.open(raw_log_file, mode="a") as writer:
            writer.write({
                "job_id": int(job_id),
                "workflow_run_id": int(workflow_run_id),
                "last_fail_sha": last_fail_sha,
                "last_fail_completed_at": str(completed_at),
                #"workflow_created_at": str(workflow_created_at),
                "log_age_days_at_fetch": age_days,
                "raw_log": raw_log,
            })

        return filtered, logs_url, "ok"

    except requests.HTTPError as e:
        code = e.response.status_code if e.response is not None else "unknown"
        body = e.response.text[:500] if e.response is not None else ""

        print(f"  HTTPError code={code}")
        print(f"  body={body}")
        
        with jsonlines.open(raw_log_file, mode="a") as writer:
            writer.write({
                "job_id": int(job_id),
                "workflow_run_id": int(workflow_run_id),
                "last_fail_sha": last_fail_sha,
                "last_fail_completed_at": str(completed_at),
                #"workflow_created_at": str(workflow_created_at),
                "log_age_days_at_fetch": age_days,
                "raw_log": "Error fetching log",
            })

        if code in {404, 410}:
            status = f"log_unavailable_or_expired_http_{code}"
        else:
            status = f"http_error_{code}"

        return None, None, status

    except Exception as e:
        with jsonlines.open(raw_log_file, mode="a") as writer:
            writer.write({
                "job_id": int(job_id),
                "workflow_run_id": int(workflow_run_id),
                "last_fail_sha": last_fail_sha,
                "last_fail_completed_at": str(completed_at),
                #"workflow_created_at": str(workflow_created_at),
                "log_age_days_at_fetch": age_days,
                "raw_log": "Error fetching log",
            })
        return None, None, f"log_fetch_error: {e}"

In [ ]:
def process_transition_row(
    row,
    workflow_run_id,
    job_id,
    last_fail_sha,
    raw_log_file: str,
    completed_at,
) -> dict:
    """
    Processes a single transition row: fetches the workflow run, finds the job,
    downloads + filters the log, and appends the raw log to `raw_log_file` (JSONL).
 
    Returns a result dict ready to append to the results list.
    """
    result = {
        "track_id": row.get("track_id"),
        "last_fail_commit":      row.get("last_fail_sha"),
        "fix_commit":            row.get("fix_sha"),
        "last_fail_workflow_run_id":       workflow_run_id,
        "last_fail_job_id":                job_id,
        "last_fail_completed_at":           row.get("last_fail_completed_at"),
        "job_name":              row.get("job_name"),
        "last_fail_workflow_status":       None,
        "last_fail_workflow_conclusion":   None,
        "last_fail_job_status":            None,
        "last_fail_job_conclusion":        None,
        "last_fail_job_logs_url":          None,
        "last_fail_job_log_text":          None,
        "last_fail_log_fetch_status":      "ok",
        'last_fail_workflow_url':      row.get("last_fail_workflow_url"),
        'last_fail_job_url':    row.get("last_fail_job_url"),
    }

 
    # ── fetch workflow run ────────────────────────────────────────────────
    workflow_run = None
    if pd.notna(workflow_run_id):
        workflow_run = repo.get_workflow_run(int(workflow_run_id))
        result["last_fail_workflow_status"]     = getattr(workflow_run, "status", None)
        result["last_fail_workflow_conclusion"] = getattr(workflow_run, "conclusion", None)
    else:
        result["last_fail_log_fetch_status"] = "missing_workflow_run_id"
 
    # ── find job ──────────────────────────────────────────────────────────
    job = None
    if pd.notna(job_id) and workflow_run is not None:
        target_job_id = int(job_id)
        for candidate in workflow_run.jobs():
            if candidate.id == target_job_id:
                job = candidate
                break
        if job is None:
            result["last_fail_log_fetch_status"] = "job_not_found"
    elif result["last_fail_log_fetch_status"] == "ok":
        result["last_fail_log_fetch_status"] = "missing_job_id_or_workflow_run"
 
    if job is not None:
        result["last_fail_job_status"]     = getattr(job, "status", None)
        result["last_fail_job_conclusion"] = getattr(job, "conclusion", None)
 
    # ── fetch + filter + store log ────────────────────────────────────────
    if job is not None:
        filtered, logs_url, fetch_status = fetch_job_log(
            job,
            job_id,
            workflow_run_id,
            last_fail_sha,
            raw_log_file,
            completed_at,
        )
        result["last_fail_job_log_text"]     = filtered
        result["last_fail_job_logs_url"]     = logs_url
        result["last_fail_log_fetch_status"] = fetch_status
 
    return result

In [ ]:
results = []

for enum_idx, (idx, row) in enumerate(transitions_df.iterrows(), start=1):

    workflow_run_id   = row.get("last_fail_workflow_run_id")
    job_id            = row.get("last_fail_job_id")
    last_fail_commit = row.get("last_fail_sha")
    fix_commit        = row.get("fix_sha")
    ##first_fail_date   = pd.to_datetime(row.get("first_fail_date"), errors="coerce")

    print(
        f"[{enum_idx}/{len(transitions_df)}] "
        f"{row.get('job_name', '')} | "
        f"{str(last_fail_commit)[:10]} -> {str(fix_commit)[:10]}"
    )

    try:
        result = process_transition_row(
            row=row,
            workflow_run_id=workflow_run_id,
            job_id=job_id,
            last_fail_sha=last_fail_commit,
            raw_log_file=RAW_LOG_FILE,
            completed_at=row.get("last_fail_completed_at"),
        )
    except Exception as e:
        print(f"  Error on row {idx}: {e}")
        result = {
            "track_id": row.get("track_id"),
            "last_fail_commit":               row.get("last_fail_sha"),
            "fix_commit":                     row.get("fix_sha"),
            "last_fail_workflow_run_id":      workflow_run_id,
            "last_fail_job_id":               job_id,
            "last_fail_completed_at":         row.get("last_fail_completed_at"),
            "job_name":                       row.get("job_name"),
            "last_fail_workflow_status":      None,
            "last_fail_workflow_conclusion":  None,
            "last_fail_job_status":           None,
            "last_fail_job_conclusion":       None,
            "last_fail_job_logs_url":         None,
            "last_fail_job_log_text":         None,
            "last_fail_log_fetch_status":     f"error: {e}",
            "last_fail_workflow_url":         row.get("last_fail_workflow_url"),
            "last_fail_job_url":              row.get("last_fail_job_url"),
        }
    print("status: " + result["last_fail_log_fetch_status"])
    results.append(result)
    time.sleep(REQUEST_DELAY)

    # ── checkpoint ────────────────────────────────────────────────────────
    if (enum_idx) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f"  Checkpoint saved at row {idx + 1} → {OUTPUT_FILE}")

# ── final save ────────────────────────────────────────────────────────────────
pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
print(f"Done — {len(results)} rows saved to {OUTPUT_FILE}")

In [ ]:
### using a loop to batch process the log download, because some logs are taking long time

In [ ]:
BATCH_CONFIG = {
    1: {"start": 0, "end": 700},
    2: {"start": 700, "end": 1400},
    3: {"start": 1400, "end": 2100},
    4: {"start": 2100, "end": None},
}
def append_results(results, output_file):
    if not results:
        return

    batch_out = pd.DataFrame(results)

    batch_out.to_csv(
        output_file,
        mode="a",
        header=not os.path.exists(output_file),
        index=False,
    )

In [ ]:
def load_existing_processed_keys(output_file):
    if not os.path.exists(output_file):
        return set()

    old = pd.read_csv(
        output_file,
        usecols=[
            "last_fail_workflow_run_id",
            "last_fail_job_id",
        ],
    )

    return set(
        zip(
            old["last_fail_workflow_run_id"].astype(str),
            old["last_fail_job_id"].astype(str),
        )
    )

In [ ]:
def process_log_batch(batch_id, transitions_df):
    cfg = BATCH_CONFIG[batch_id]

    output_file = f"pytorch_transition_withlog_0928_logbatch{batch_id}.csv"
    raw_log_file = f"pytorch_transition_raw_logs_0928_logbatch{batch_id}.jsonl"

    batch_df = transitions_df.iloc[cfg["start"]:cfg["end"]].copy()

    processed_keys = load_existing_processed_keys(output_file)
    results = []

    print("\n" + "=" * 80)
    print(f"Starting batch {batch_id}")
    print(f"Rows: {len(batch_df)}")
    print(f"Output: {output_file}")
    print(f"Raw logs: {raw_log_file}")
    print(f"Already processed: {len(processed_keys)}")
    print("=" * 80)

    for enum_idx, (idx, row) in enumerate(batch_df.iterrows(), start=1):
        workflow_run_id = row.get("last_fail_workflow_run_id")
        job_id = row.get("last_fail_job_id")
        last_fail_commit = row.get("last_fail_sha")
        fix_commit = row.get("fix_sha")

        key = (str(workflow_run_id), str(job_id))

        if key in processed_keys:
            print(
                f"[batch {batch_id} | {enum_idx}/{len(batch_df)}] "
                f"already processed job_id={job_id}; skipping"
            )
            continue

        print(
            f"[batch {batch_id} | {enum_idx}/{len(batch_df)} | original idx {idx}] "
            f"{row.get('job_name', '')} | "
            f"{str(last_fail_commit)[:10]} -> {str(fix_commit)[:10]}"
        )

        try:
            result = process_transition_row(
                row=row,
                workflow_run_id=workflow_run_id,
                job_id=job_id,
                last_fail_sha=last_fail_commit,
                raw_log_file=raw_log_file,
                completed_at=row.get("last_fail_completed_at"),
            )

        except Exception as e:
            print(f"  Error on original row {idx}: {e}")

            result = {
                "track_id": row.get("track_id"),
                "last_fail_commit": row.get("last_fail_sha"),
                "fix_commit": row.get("fix_sha"),
                "last_fail_workflow_run_id": workflow_run_id,
                "last_fail_job_id": job_id,
                "last_fail_completed_at": row.get("last_fail_completed_at"),
                "job_name": row.get("job_name"),
                "last_fail_workflow_status": None,
                "last_fail_workflow_conclusion": None,
                "last_fail_job_status": None,
                "last_fail_job_conclusion": None,
                "last_fail_job_logs_url": None,
                "last_fail_job_log_text": None,
                "last_fail_log_fetch_status": f"error: {type(e).__name__}: {e}",
                "last_fail_workflow_url": row.get("last_fail_workflow_url"),
                "last_fail_job_url": row.get("last_fail_job_url"),
            }

        result["source_batch_id"] = batch_id
        result["source_transition_index"] = idx

        print("status: " + result["last_fail_log_fetch_status"])

        results.append(result)
        processed_keys.add(key)

        time.sleep(REQUEST_DELAY)

        if len(results) % SAVE_EVERY == 0:
            append_results(results, output_file)
            results = []
            print(f"  Checkpoint appended -> {output_file}")

    append_results(results, output_file)
    results = []

    print(f"Finished batch {batch_id}: {output_file}")

In [ ]:
for batch_id in [1, 2, 3, 4]:
    process_log_batch(batch_id, transitions_df)

    print("Resting 5 minutes before next batch...")
    time.sleep(5 * 60)

In [ ]:
withlog_files = [
    f"pytorch_transition_withlog_0928_logbatch{batch_id}.csv"
    for batch_id in [1, 2, 3, 4]
]

withlog_parts = []

for path in withlog_files:
    if os.path.exists(path):
        part = pd.read_csv(path)
        part["source_log_batch_file"] = path
        withlog_parts.append(part)
    else:
        print(f"Missing file: {path}")

all_withlog = pd.concat(
    withlog_parts,
    ignore_index=True,
)

withlog_key = [
    "track_id",
    "last_fail_commit",
    "fix_commit",
    "last_fail_workflow_run_id",
    "last_fail_job_id",
]

print("Rows before dedup:", len(all_withlog))
print(
    "Duplicate rows:",
    all_withlog.duplicated(subset=withlog_key, keep=False).sum(),
)

all_withlog = (
    all_withlog
    .drop_duplicates(subset=withlog_key, keep="last")
    .reset_index(drop=True)
)

print("Rows after dedup:", len(all_withlog))

all_withlog.to_csv(
    "pytorch_transition_withlog_0928_run2.csv",
    index=False,
)

In [ ]:
transitions_with_log = pd.read_csv("pytorch_transition_withlog_0928_run2.csv")

In [ ]:
####Reusing logs from run 1 
withlog_files = [
    "pytorch_transition_withlog_0928_part1.csv",
    "pytorch_transition_withlog_0928_part2.csv",
    "pytorch_transition_withlog_0928_part3.csv",
    "pytorch_transition_withlog_0928_part4.csv",
]

all_withlog = pd.concat(
    [pd.read_csv(path) for path in withlog_files],
    ignore_index=True,
)
len(all_withlog)

In [ ]:
all_withlog.columns

In [ ]:
withlog_key = [
    "track_id",
    "last_fail_commit",
    "fix_commit",
    "last_fail_workflow_run_id",
    "last_fail_job_id",
]
print("Rows before dedup:", len(all_withlog))
print("Duplicate rows:", all_withlog.duplicated(subset=withlog_key, keep=False).sum())

In [ ]:
all_withlog.to_csv("pytorch_transition_withlog_0928_run2.csv", index=False)

In [ ]:
transitions_check = transitions_df.merge(
    all_withlog[withlog_key + ["last_fail_log_fetch_status"]],
    left_on=transition_key,
    right_on=withlog_key,
    how="left",
    indicator=True,
)

print(transitions_check["_merge"].value_counts())

In [ ]:
missing_from_withlog = transitions_check[
    transitions_check["_merge"].eq("left_only")
].copy()

print("Missing transitions:", len(missing_from_withlog))

In [ ]:
complete_withlog = transitions_df.merge(
    all_withlog.drop(columns=["track_id"], errors="ignore"),
    left_on=[
        "last_fail_sha",
        "fix_sha",
        "last_fail_workflow_run_id",
        "last_fail_job_id",
    ],
    right_on=[
        "last_fail_commit",
        "fix_commit",
        "last_fail_workflow_run_id",
        "last_fail_job_id",
    ],
    how="left",
    indicator=True,
)
complete_withlog.head()

In [ ]:
complete_withlog.to_csv("pytorch_transition_withlog_0928_run12merge.csv", index=False)

In [ ]:
transitions_with_log = all_withlog.copy()

In [ ]:
#transitions_with_log = pd.DataFrame(results)

In [ ]:
len(transitions_with_log)

In [ ]:
transitions_with_log

In [ ]:
test_jobs = transitions_with_log["last_fail_job_url"]
print(test_jobs[86])
##maunally confirmed that logs are unavailable

In [ ]:
###added a fallback for log unavaialble cases --> we fetch the annotations 

In [ ]:
HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {TOKEN}",
    "X-GitHub-Api-Version": "2022-11-28",
}


def get_job_metadata(job_id, max_retries=5, base_sleep=10):
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/actions/jobs/{int(job_id)}"

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.json(), "ok"

        except requests.RequestException as e:
            if attempt == max_retries:
                return None, f"job_metadata_error: {type(e).__name__}: {e}"

            time.sleep(base_sleep * attempt)


def fetch_check_run_annotations(check_run_url, max_retries=5, base_sleep=10):
    if not check_run_url:
        return [], "missing_check_run_url"

    annotations = []
    page = 1

    while True:
        url = f"{check_run_url}/annotations"
        params = {"per_page": 100, "page": page}

        for attempt in range(1, max_retries + 1):
            try:
                r = requests.get(
                    url,
                    headers=HEADERS,
                    params=params,
                    timeout=30,
                )

                if r.status_code == 404:
                    return annotations, "check_run_annotations_not_found_404"

                if r.status_code == 410:
                    return annotations, "check_run_annotations_gone_410"

                r.raise_for_status()
                batch = r.json()

                if not batch:
                    return annotations, "ok"

                annotations.extend(batch)

                if len(batch) < 100:
                    return annotations, "ok"

                page += 1
                break

            except requests.RequestException as e:
                if attempt == max_retries:
                    return annotations, f"annotations_fetch_error: {type(e).__name__}: {e}"

                time.sleep(base_sleep * attempt)

In [ ]:
def annotation_text(annotations):
    if not annotations:
        return ""

    lines = []

    for ann in annotations:
        path = ann.get("path")
        start_line = ann.get("start_line")
        end_line = ann.get("end_line")
        level = ann.get("annotation_level")
        title = ann.get("title")
        message = ann.get("message")
        raw_details = ann.get("raw_details")

        location = path or "<no_path>"
        if start_line is not None:
            location += f":{start_line}"
            if end_line is not None and end_line != start_line:
                location += f"-{end_line}"

        parts = [
            str(level) if level else None,
            location,
            str(title) if title else None,
            str(message) if message else None,
        ]

        line = " | ".join(part for part in parts if part)
        lines.append(line)

        if raw_details:
            lines.append(str(raw_details))

    return "\n".join(lines)

In [ ]:
ANNOTATIONS_FILE = "job_annotations_part4.jsonl"

failed_log_rows = transitions_with_log[
    transitions_with_log["last_fail_log_fetch_status"].ne("ok")
].copy()

annotation_rows = []

for i, row in enumerate(failed_log_rows.itertuples(index=False), 1):
    job_id = row.last_fail_job_id
    workflow_run_id = row.last_fail_workflow_run_id
    last_fail_sha = row.last_fail_commit

    job_meta, job_meta_status = get_job_metadata(job_id)

    if job_meta is None:
        annotations = []
        ann_status = job_meta_status
        check_run_url = None
    else:
        check_run_url = job_meta.get("check_run_url")
        annotations, ann_status = fetch_check_run_annotations(check_run_url)

    annotation_rows.append({
        "job_id": int(job_id) if pd.notna(job_id) else None,
        "workflow_run_id": int(workflow_run_id) if pd.notna(workflow_run_id) else None,
        "last_fail_sha": last_fail_sha,
        "job_metadata_status": job_meta_status,
        "check_run_url": check_run_url,
        "annotation_fetch_status": ann_status,
        "annotation_count": len(annotations),
        "annotation_text": annotation_text(annotations),
        "annotations": annotations,
    })

    with jsonlines.open(ANNOTATIONS_FILE, mode="a") as writer:
        writer.write(annotation_rows[-1])

    print(
        f"[{i}/{len(failed_log_rows)}] "
        f"job_id={job_id} annotations={len(annotations)} status={ann_status}"
    )

    time.sleep(1)

annotations_df = pd.DataFrame(annotation_rows)

In [ ]:
len(annotations_df)

In [ ]:
annotations_df.head()

In [ ]:
annotations_df.to_csv("transitions_nolog_with_annotations_part4.csv")

In [ ]:
###fallback metadata.
transitions_with_log_and_annotations = transitions_with_log.merge(
    annotations_df[
        [
            "job_id",
            "job_metadata_status",
            "check_run_url",
            "annotation_fetch_status",
            "annotation_count",
            "annotation_text",
        ]
    ],
    left_on="last_fail_job_id",
    right_on="job_id",
    how="left",
)

In [ ]:
len(transitions_with_log_and_annotations)

In [ ]:
transitions_with_log_and_annotations.to_csv("transitions_withlog_and_annotations_part4.csv")

In [ ]:
annotation_patterns = {
    "failure annotation": r"(?im)^failure\s*\|",
    "warning annotation": r"(?im)^warning\s*\|",
    "error:": r"(?i)\berror:",
    "Error": r"(?i)\berror\b",
    "FAILED test": r"(?i)\bFAILED\s+test\b",
    "FAILED": r"(?i)\bFAILED\b",
    "AssertionError": r"(?i)\bAssertionError\b",
    "assert": r"(?i)\bassert\b",
    "Traceback": r"(?i)\bTraceback\b",
    "ERRORS": r"(?i)\bERRORS\b",
    "tests/": r"(?i)tests/",
    "test": r"(?i)\btest\b",
    "File traceback": r'(?i)\bFile\s+"?[^",\n]+"?,\s+line\s+\d+',
    "exit code": r"(?i)Process completed with exit code \d+",
    "timeout": r"(?i)\btimeout\b|\btimed out\b",
    "cancelled/canceled": r"(?i)\bcancelled\b|\bcanceled\b",
    "runner/container": r"(?i)custom container|self hosted runner|pod failed|jobPod|phase status Pending",
    "download/network": r"(?i)failed to download|codeload\.github\.com|HttpClient\.Timeout|TLS handshake timeout",
    "Node warning": r"(?i)Node\.js 20 is deprecated",
    "test_step_timeout": r"(?i)The action 'Test' has timed out",
    "test_config_warning": r"(?i)Gathered no stats from artifacts.*test config",
    "pytest_signal": r"(?i)\bpytest\b|tests/",
    "assertion_signal": r"(?i)\bAssertionError\b|\bassert\b",
}

In [ ]:
annotation_col = "annotation_text"

df_ann = annotations_df[annotations_df[annotation_col].notna() & annotations_df[annotation_col].astype(str).str.strip().ne("")].copy()

for label, pat in annotation_patterns.items():
    count = df_ann[annotation_col].str.contains(
        pat,
        regex=True,
        na=False,
    ).sum()

    pct = count / len(df_ann) * 100 if len(df_ann) else 0

    print(
        f"  {label:<22} {count:>4} / {len(df_ann)}  ({pct:.0f}%)"
    )

In [ ]:
##annitations are not used as evidence for later pipeline because they are too board. annotations usually point to the GitHub Actions wrapper step, not the underlying failing assertion or traceback. 
#So they would add noise and could cause the model to overfit on infra symptoms.

In [ ]:
transitions_with_log_valid = transitions_with_log[transitions_with_log["last_fail_job_log_text"].notna() & (transitions_with_log["last_fail_job_log_text"] != "")]

patterns = {
    "##[error]":     r"##\[error\]",
    "error:":        r"error:",
    "FAILED test":   r"FAILED\s+test",
    "FAILED":        r"FAILED",
    "AssertionError": r"AssertionError",
    "ERRORS":        r"\bERRORS\b",
    "tests/":        r"tests/",
    "test":          r"(?i).*test.*",
    "File":          r'(?i)\bFile\s+"?[^",\n]+"?,\s+line\s+\d+',
}

print(f"Total rows with log: {len(transitions_with_log_valid)}\n")

In [ ]:
for label, pat in patterns.items():
    count = transitions_with_log_valid["last_fail_job_log_text"].str.contains(pat, case=False, regex=True, na=False).sum()
    pct   = count / len(transitions_with_log_valid) * 100
    print(f"  {label:<20} {count:>4} / {len(transitions_with_log_valid)}  ({pct:.0f}%)")

In [ ]:
assertion_mask = transitions_with_log_valid["last_fail_job_log_text"].str.contains(r"AssertionError", case=False, regex=True, na=False)
transitions_with_assertion = transitions_with_log_valid[assertion_mask]
print(f"Rows with 'AssertionError': {len(transitions_with_assertion)}\n")

In [ ]:
transitions_with_assertion.columns

In [ ]:
def extract_assertion_errors(log_text: str) -> str:
    """
    Extracts unique AssertionError lines only (no context window needed for classifier).
    Deduplicates by stripping timestamps before comparison.
    """
    if pd.isna(log_text) or not str(log_text).strip():
        return ""
    
    lines = str(log_text).splitlines()
    
    # Extract only AssertionError lines
    assertion_lines = [
        line for line in lines
        if re.search(r"AssertionError", line, re.IGNORECASE)
    ]
    
    # Dedup: strip leading timestamp (e.g. "2026-07-10T21:52:20.8638993Z ") before comparing
    seen = set()
    deduped = []
    for line in assertion_lines:
        # Remove timestamp prefix for comparison, keep original for output
        normalized = re.sub(r"^\d{4}-\d{2}-\d{2}T[\d:.]+Z\s*", "", line).strip()
        if normalized not in seen:
            seen.add(normalized)
            deduped.append(line)
    
    return "\n".join(deduped)

In [ ]:
transitions_with_assertion = transitions_with_assertion.copy()  # avoid SettingWithCopyWarning
transitions_with_assertion["assertion_error_lines"] = transitions_with_assertion["last_fail_job_log_text"].apply(extract_assertion_errors)

In [ ]:
pytest_node_re = re.compile(
    r"(?i)\b[\w./-]*test[\w./-]*\.py::[\w:.\[\]-]+"
)

failed_node_re = re.compile(
    r"(?i)\bFAILED\s+(?:\[[^\]]+\]\s+)?"
    r"([\w./-]*test[\w./-]*\.py::[\w:.\[\]-]+)"
)

assert_re = re.compile(r"AssertionError")

def extract_failure_context(log_text, window=10):
    lines = str(log_text or "").splitlines()

    assertion_idxs = [
        i for i, line in enumerate(lines)
        if assert_re.search(line)
    ]

    selected = []
    seen = set()

    for idx in assertion_idxs:
        start = max(0, idx - window)
        end = min(len(lines), idx + window + 1)

        for j in range(start, end):
            if j not in seen:
                seen.add(j)
                selected.append(lines[j])

    failed_tests = []
    pytest_tests = []

    for line in selected:
        for m in failed_node_re.finditer(line):
            failed_tests.append(m.group(1))

        for m in pytest_node_re.finditer(line):
            pytest_tests.append(m.group(0))

    return {
        "failed_tests": "; ".join(sorted(set(failed_tests))),
        "pytest_tests": "; ".join(sorted(set(pytest_tests))),
        #"assertion_context": "\n".join(selected),
    }

In [ ]:
extracted = transitions_with_assertion["last_fail_job_log_text"].apply(
    extract_failure_context
)

extracted_df = pd.DataFrame(extracted.tolist(), index=transitions_with_assertion.index)

transitions_with_failure_context = pd.concat(
    [transitions_with_assertion, extracted_df],
    axis=1,
)
transitions_with_failure_context.head(10)

In [ ]:
transitions_with_failure_context.to_csv("transitions_with_failure_context_0928_run2.csv")

In [ ]:
transitions_with_failure_context.columns

In [ ]:
transitions_with_failure_context.head()

In [ ]:
##run3 reuse 
transitions_with_failure_context = pd.read_csv("transitions_with_failure_context_0928_run2.csv")
transitions_with_failure_context.head(3)

In [ ]:
##run4 reuse 
transitions_with_failure_context = pd.read_csv("transitions_with_failure_context_0928_run2.csv")

In [ ]:
transitions_with_failure_context.columns

In [ ]:
len(transitions_with_failure_context)

In [ ]:
from google import genai
from google.genai import types

gemini_35_response = []
gemini_35_thoughts = []

client = genai.Client(api_key=gemini_key)

In [ ]:
def call_gemini(prompt):
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            thinking_config=types.ThinkingConfig(include_thoughts=False),
        ),
    )
    return response.text if response.text else "No response from the model."

In [ ]:
def parse_field(text, field_name):
    for line in str(text).splitlines():
        if line.lower().startswith(field_name.lower() + ":"):
            return line.split(":", 1)[1].strip()
    return None

In [ ]:
def build_assertion_classify_prompt(row):
    return f"""
You are reviewing AssertionError lines extracted from a PyTorch CI log.

Goal: Classify the errors into exactly one category:

- code_fix_expected: deterministic logic failure with a clear expected vs actual value mismatch — wrong count, wrong shape, wrong keys, wrong type, wrong behavior. A code change is expected to fix this.
- hardware_error: the failure is related to hardware, device, or platform-specific behavior — includes numerical precision differences, CUDA/ROCm/CPU backend differences, GPU-specific errors, or results that vary by hardware.
- xpass: unexpected test pass — a test marked as xfail unexpectedly passed, fix is removing it from the xfail list.
- other: infrastructure failure, timeout, OOM, setup error, or anything not clearly a test logic failure.

Rules:
- If multiple AssertionError types are present, classify by the dominant/most specific one.
- If the message is empty or contains no meaningful signal, output: other

Output format:
Assertion_Error_Category: <code_fix_expected|hardware_error|xpass|other>
Reason: 1-3 sentences of reason for classification

AssertionError lines:
{row.get("assertion_error_lines")}
"""

In [ ]:
gemini_responses = []
assertion_categories = []
assertion_reasons = []

for enum_idx, (idx, row) in enumerate(transitions_with_failure_context.iterrows(), start=1):
    prompt = build_assertion_classify_prompt(row)
    try:
        response_text = call_gemini(prompt)
    except Exception as e:
        response_text = f"ERROR: {e}"

    print(f"[{enum_idx}/{len(transitions_with_failure_context)}] idx={idx}\n{response_text}\n")

    gemini_responses.append(response_text)
    assertion_categories.append(parse_field(response_text, "Assertion_Error_Category"))
    assertion_reasons.append(parse_field(response_text, "Reason"))

In [ ]:
transitions_with_failure_context["assertion_category"]      = assertion_categories
transitions_with_failure_context["assertion_category_reason"] = assertion_reasons

In [ ]:
transitions_with_failure_context["assertion_category"].value_counts()

In [ ]:
transitions_with_failure_context.to_csv("transitions_with_failure_gemini_0928_run4.csv")

In [ ]:
transitions_with_code_fix_expected = transitions_with_failure_context[
    transitions_with_failure_context["assertion_category"] == "code_fix_expected"
]
len(transitions_with_code_fix_expected)

In [ ]:
transitions_with_code_fix_expected.to_csv("transitions_with_code_fix_expected_0928_run4.csv")

In [ ]:
transitions_with_code_fix_expected.columns

In [ ]:
transitions_with_code_fix_expected = transitions_with_code_fix_expected.reset_index(
    names="transition_id"
)
transitions_with_code_fix_expected.columns

In [ ]:
transitions_with_code_fix_expected = transitions_with_code_fix_expected.drop(
    columns=[
        "transition_source_index",
        "agent_input_row_index",
    ],
    errors="ignore",
)

In [ ]:
transitions_with_code_fix_expected.columns

In [ ]:
transitions_with_code_fix_expected = transitions_with_code_fix_expected.copy()

transitions_with_code_fix_expected = (
    transitions_with_code_fix_expected
    .reset_index()
    .rename(columns={"index": "transition_source_index"})
    .reset_index()
    .rename(columns={"index": "agent_input_row_index"})
)
transitions_with_code_fix_expected.columns

In [ ]:
transitions_with_code_fix_expected.columns

In [ ]:
transitions_with_code_fix_expected.head(14)

In [ ]:
len(transitions_with_code_fix_expected)

In [ ]:
#transitions_with_code_fix_expected = pd.read_csv("transitions_with_code_fix_expected_0928_part3.csv")

In [ ]:
transition_1k_df = pd.read_csv("track_based_transitions_enriched.csv")
len(transition_1k_df)

In [ ]:
previous_rows = pd.read_csv("code_fix_expected_all_run3.csv")
len(previous_rows)

In [ ]:
CODE_EXPECTED_WITH_TEST_CONTEXT_SINGLE_COMMIT_WINDOW_SKILL = """
You are identifying the PyTorch GitHub commit that best explains recovery from a CI log error that was already classified as code_fix_expected by a prior log-only LLM pass.

Task:
Given one Git commit recovery window and enriched CI log error context, find whether one best single commit in the window plausibly fixed the product/source behavior shown by the error. If no commit explains the recovery, classify why no suitable product-code fix commit was found.

Important context:
- The input error was already filtered/classified from extracted CI logs as code_fix_expected.
- Treat code_fix_expected as a log-level signal: the extracted error appears to show a deterministic semantic/assertion mismatch where a code fix is plausible.
- Prefer commits that change code behavior over commits that only change tests, skips, xfails, CI, infra, dependencies, runners, or environment.
- Use extracted test context to avoid matching a commit that fixes a different component, operator, or test than the one shown in the log.

The window type is:
- recovery_window: commits may contain the product-code fix that recovered the error

Interpretation of "explain":
- recovery_window: find commit evidence that plausibly fixed the deterministic behavior mismatch shown in one or more provided log errors

Required inputs:
- window_start_commit/window start date
- window_end_commit/window end date
- assertion_error_lines: extracted AssertionError lines from the failing CI job log
- failed_tests: extracted FAILED ... test_file.py::test_name lines near AssertionError lines, if present

Required procedure:
1. Use GitHub MCP to find PyTorch commits whose commit dates fall within window_start_commit and window_end_commit.
   window_start_commit is exclusive. window_end_commit is inclusive. Only analyze commits after window_start_commit and up to window_end_commit.
2. Inspect commits greedily from earliest to latest.
3. For each candidate, inspect commit title/message, changed files, and patch/diff.
4. Identify the core code-fix signal(s) from assertion_error_lines, failed_tests:
   failing test name, assertion mismatch, expected/actual values, shape/count/key/type mismatch, operator, module, stack path, API behavior, exception/warning mismatch, dtype/device behavior, or other deterministic semantic mismatch.
5. A commit does not need to explain every error in the log. It can be a match if it explains one provided code_fix_expected error.
6. Prefer patch/file evidence: changed source path, implementation logic, operator behavior, failing test path, stack path, component, API, dtype/shape/count handling, or error behavior named or strongly implied by the extracted log context.
7. Commit title/message can support a match, but do not return a match based on generic repair wording alone.
8. Stop once one best strong_patch_match is found. If only weaker evidence is found, continue until either stronger evidence appears or the window is exhausted.
9. Base the final decision on MCP-observed commit evidence and the provided log context.
10. Prefer tool-observed repository evidence over date proximity.

Search policy:
- First obtain or infer the ordered commit list for the window.
- Inspect commits in the required direction.
- For each commit, inspect at least the commit title/message, changed files, and changed hunks/diffs.
- Inspect the full patch and related repo context when changed files or hunks have weak relation to the extracted test/error context.
- Continue until one of these happens:
  1. A strong_patch_match is found.
  2. No stronger match is likely after inspecting the relevant window.
  3. A clear indirect_recovery or out_of_repo_possible classification is justified.
  4. The whole window is exhausted.
- Do not return no_match merely because the first few commits are weak.
- If the window is very large and tool/context limits prevent full inspection, report that explicitly in stop_reason.

Match categories:
- strong_patch_match: A single commit directly explains one or more code_fix_expected errors through clear patch, changed-file, code-path, failing-test-path, or operator/module evidence. The patch changes product/source behavior or closely related implementation logic. The commit should align with the extracted failed_tests/pytest_tests when those fields are present.
- weak_patch_match: A single commit plausibly relates to one or more code_fix_expected errors through patch, changed-file, code-path, test-path, or component evidence, but the causal link is incomplete, indirect, or uncertain.
- indirect_recovery: A commit plausibly recovered the observed failure without fixing product/source behavior, such as by changing a test expectation, adding/removing skip or xfail, changing CI/config, updating dependencies, changing runners, reverting behavior, or masking the error.
- out_of_repo_possible: Evidence suggests the fix may be outside the PyTorch repository, such as runtime, driver, Docker image, machine image, upstream library, external dependency, or external repository.
- no_match: Inspected commits do not plausibly explain any provided code_fix_expected error through patch/file/test-path evidence.

Decision priority:
- Use strong_patch_match when a code patch strongly explains the observed deterministic mismatch and aligns with the extracted test/error context.
- Use weak_patch_match when the files or hunks are relevant but the causal connection remains uncertain.
- Use indirect_recovery when the relevant commit changes tests, skips, xfails, CI, configuration, dependency/environment, or other non-product-code behavior rather than fixing the underlying implementation.
- Use out_of_repo_possible when repository patches do not explain the issue and evidence points to runtime/driver/image/environment/upstream behavior.
- Use no_match when the inspected window does not contain a plausible product-code fix or recovery commit.
- Do not use any match category merely because the commit is close in time.

Indirect recovery types:
- test_expectation_update
- test_skip_or_xfail
- revert
- ci_or_infra_change
- dependency_or_environment_change
- config_or_runner_change
- other
- none

Traceability requirements:
- Track each candidate commit you inspect.
- num_commits_inspected must count the number of candidate commits inspected.
- commits_inspected must list those commit SHAs in the order inspected.
- related_commit_sha must be filled for strong_patch_match, weak_patch_match, and commit-specific indirect_recovery.
- related_commit_sha should be none for no_match and out_of_repo_possible unless a specific commit supports the classification.
- stop_reason should explain why you stopped: strong_patch_match_found, best_weak_match_found, indirect_recovery_found, out_of_repo_possible, window_exhausted_no_match, MCP unavailable, tool/context limit, or tool_error.

Reason requirements:
- Keep the reason short.
- Mention the specific failed_tests or pytest_tests signal used when present.
- Mention the most important 1-3 inspected paths or repo context items.
- Mention whether the evidence came mainly from patch/file evidence, commit title/message support, or both.
- If the commit matches only one error among multiple visible errors, say which one.
- If failed_tests/pytest_tests were empty or too generic, say that the match relied on assertion_error_lines/assertion_context only.
- If MCP evidence was not used, say so explicitly.
- When no_match is identified, briefly explain why no suitable commit was found in the provided window.

Output format:
Return exactly these lines and nothing else:

match_category: strong_patch_match/weak_patch_match/indirect_recovery/out_of_repo_possible/no_match
related_commit_sha: commit sha or none
related_commit_url: commit url or none
confidence: high/medium/low
indirect_recovery_type: test_expectation_update/test_skip_or_xfail/revert/ci_or_infra_change/dependency_or_environment_change/config_or_runner_change/other/none
num_commits_inspected: integer
commits_inspected: semicolon-separated commit shas or none
stop_reason: short explanation
no_match_reason_type: window_commits_unrelated/weak_error_signal/possible_flaky_or_transient/non_product_code_behavior_change/window_too_limited/insufficient_mcp_evidence/other/none
reason: short explanation with key evidence
""".strip()

In [ ]:
def extract_trace_and_tool_items(result):
    trace_items = []
    tool_items = []

    for item in result.new_items:
        item_type = type(item).__name__

        if hasattr(item, "model_dump"):
            item_data = item.model_dump()
        else:
            item_data = {"repr": str(item)}

        trace_item = {
            "item_type": item_type,
            "data": item_data,
        }

        trace_items.append(trace_item)

        if "Tool" in item_type or "tool" in item_type:
            tool_items.append(trace_item)

    return trace_items, tool_items

In [ ]:
def build_row_prompt_commit_window(row):
    return f"""
You are receiving one PyTorch commit window and corresponding test-failure errors fall within that window.

Repository:
https://github.com/pytorch/pytorch

window_type: recovery_window

window_start_commit:
{row.get("last_fail_commit", "")}

window_end_commit:
{row.get("fix_commit", "")}

assertion_error_lines:
```text
{row.get("assertion_error_lines", "")}

failed_tests:
```text
{row.get("failed_tests", "")}
""".strip()

In [ ]:
def parse_agent_output_commit_window(text):
    parsed = {
        "match_category": "",
        #"match_unit": "",
        "related_commit_sha": "",
        "related_commit_url": "",
        "confidence": "",
        #"text_evidence_type": "",
        "indirect_recovery_type": "",
        "num_commits_inspected": "",
        "commits_inspected": "",
        "stop_reason": "",
        "no_match_reason_type": "",
        "reason": "",
        "agent_raw_output": text,
    }

    for line in text.splitlines():
        line = line.strip()

        if line.startswith("match_category:"):
            parsed["match_category"] = line.split(":", 1)[1].strip()
        #elif line.startswith("match_unit:"):
        #    parsed["match_unit"] = line.split(":", 1)[1].strip()
        elif line.startswith("related_commit_sha:"):
            parsed["related_commit_sha"] = line.split(":", 1)[1].strip()
        elif line.startswith("related_commit_url:"):
            parsed["related_commit_url"] = line.split(":", 1)[1].strip()
        elif line.startswith("confidence:"):
            parsed["confidence"] = line.split(":", 1)[1].strip()
        #elif line.startswith("text_evidence_type:"):
        #    parsed["text_evidence_type"] = line.split(":", 1)[1].strip()   
        elif line.startswith("indirect_recovery_type:"):
            parsed["indirect_recovery_type"] = line.split(":", 1)[1].strip()
        elif line.startswith("num_commits_inspected:"):
            parsed["num_commits_inspected"] = line.split(":", 1)[1].strip()
        elif line.startswith("commits_inspected:"):
            parsed["commits_inspected"] = line.split(":", 1)[1].strip()
        elif line.startswith("stop_reason:"):
            parsed["stop_reason"] = line.split(":", 1)[1].strip()
        elif line.startswith("no_match_reason_type:"):
            parsed["no_match_reason_type"] = line.split(":", 1)[1].strip()   
        elif line.startswith("reason:"):
            parsed["reason"] = line.split(":", 1)[1].strip()

    return parsed

In [ ]:
def build_commit_window_result_row(row, parsed_output):

    match_category = parsed_output.get("match_category", "")
    #match_unit = parsed_output.get("match_unit", "")

    related_sha = parsed_output.get("related_commit_sha", "")
    related_url = parsed_output.get("related_commit_url", "")

    text_evidence_type = parsed_output.get("text_evidence_type", "")
    indirect_recovery_type = parsed_output.get("indirect_recovery_type", "")
    no_match_reason_type = parsed_output.get("no_match_reason_type", "")

    # Backward-compatible flag.
    # If you want indirect_recovery to count as found, add it here.
    related_commit_found = (
        "true"
        if match_category in ["strong_match", "weak_match"]
        else "false"
    )

    error_inducing_commit = ""
    error_inducing_commit_url = ""
    fix_addressing_commit = ""
    fix_addressing_commit_url = ""

    #error_inducing_cumulative_start_sha = ""
    #error_inducing_cumulative_end_sha = ""
    #fix_addressing_cumulative_start_sha = ""
    #fix_addressing_cumulative_end_sha = ""

    #if window["window_type"] == "failure_window":
    #    error_inducing_commit = related_sha
    #    error_inducing_commit_url = related_url
        #error_inducing_cumulative_start_sha = cumulative_start_sha
        #error_inducing_cumulative_end_sha = cumulative_end_sha

    #elif window["window_type"] == "recovery_window":
    #    fix_addressing_commit = related_sha
    #    fix_addressing_commit_url = related_url
        #fix_addressing_cumulative_start_sha = cumulative_start_sha
        #fix_addressing_cumulative_end_sha = cumulative_end_sha

    fix_addressing_commit = related_sha
    fix_addressing_commit_url = related_url

    return {
        # Source metadata
        "source_first_bad_commit": row.get("first_bad_commit", row.get("first_fail_sha", "")),
        "source_fix_commit": row.get("fix_commit", row.get("fix_sha", "")),
        "workflow_run_id": row.get("workflow_run_id", row.get("last_fail_workflow_run_id", "")),
        "job_id": row.get("job_id", row.get("last_fail_job_id", "")),
        "transition_id": row.get("transition_id", ""),
        "track_id": row.get("track_id", ""),
        "workflow_name": row.get("workflow_name", ""),
        "job_name": row.get("job_name", ""),
        

        # Window metadata
        "window_type": "recovery_window",
        "window_start_commit": row.get("last_fail_commit", ""),
        "window_end_commit": row.get("fix_commit", ""),
        #"window_size": window.get("window_size", row.get("window_size", "")),

        # New incremental-analysis output
        "match_category": match_category,
        #"match_unit": match_unit,
        "related_commit_sha": related_sha,
        "related_commit_url": related_url,
        #"cumulative_window_start_sha": cumulative_start_sha,
       # "cumulative_window_end_sha": cumulative_end_sha,
        #"cumulative_window_commit_count": cumulative_count,
        #"indirect_recovery_type": parsed_output.get("indirect_recovery_type", ""),
        "confidence": parsed_output.get("confidence", ""),
        "text_evidence_type": text_evidence_type,
        "indirect_recovery_type": indirect_recovery_type,

        # Backward-compatible output
        "related_commit_found": related_commit_found,

        # Role-specific columns
        "error_inducing_commit": error_inducing_commit,
        "error_inducing_commit_url": error_inducing_commit_url,
        #"error_inducing_cumulative_start_sha": error_inducing_cumulative_start_sha,
        #"error_inducing_cumulative_end_sha": error_inducing_cumulative_end_sha,

        "fix_addressing_commit": fix_addressing_commit,
        "fix_addressing_commit_url": fix_addressing_commit_url,
        #"fix_addressing_cumulative_start_sha": fix_addressing_cumulative_start_sha,
        #"fix_addressing_cumulative_end_sha": fix_addressing_cumulative_end_sha,

        # Traceability
        "reason": parsed_output.get("reason", ""),
        "num_commits_inspected": parsed_output.get("num_commits_inspected", ""),
        "commits_inspected": parsed_output.get("commits_inspected", ""),
        "stop_reason": parsed_output.get("stop_reason", ""),
        "no_match_reason_type": parsed_output.get("no_match_reason_type", ""),
        "agent_raw_output": parsed_output.get("agent_raw_output", ""),
    }

In [ ]:
CHUNK_SIZE = 1

RESULTS_PARTIAL_PATH = "mcp_commit_window_agent_results_partial_code_exp_0928_run4.csv"
DEBUG_PARTIAL_PATH = "mcp_commit_window_agent_debug_logs_partial_code_exp_0928_run4.json"
RESULTS_FINAL_PATH = "mcp_commit_window_agent_analysis_code_exp_0928_run4.csv"
DEBUG_FINAL_PATH = "mcp_commit_window_agent_debug_logs_code_exp_0928_run4.json"


def save_checkpoint(outputs, debug_logs):
    pd.DataFrame(outputs).to_csv(RESULTS_PARTIAL_PATH, index=False)

    with open(DEBUG_PARTIAL_PATH, "w") as f:
        json.dump(debug_logs, f, indent=2, ensure_ascii=False)


def build_error_result_row(row, idx, error, stop_reason):
    error_type = type(error).__name__
    error_message = str(error)

    return {
        "source_row_index": int(idx),

        # Source metadata
        "source_first_bad_commit": row.get("first_bad_commit", row.get("first_fail_sha", "")),
        "source_fix_commit": row.get("fix_commit", row.get("fix_sha", "")),
        "workflow_run_id": row.get("workflow_run_id", row.get("last_fail_workflow_run_id", "")),
        "job_id": row.get("job_id", row.get("last_fail_job_id", "")),
        "transition_id": row.get("transition_id", ""),
        "track_id": row.get("track_id", ""),
        "workflow_name": row.get("workflow_name", ""),
        "job_name": row.get("job_name", ""),

        # Window metadata
        "window_type": "recovery_window",
        "window_start_commit": row.get("last_fail_commit", ""),
        "window_end_commit": row.get("fix_commit", ""),
        #"window_size": window.get("window_size", row.get("window_size", "")),
        
        # New incremental-analysis output
        "match_category": "tool_error",
        #"match_unit": "none",
        "related_commit_sha": "none",
        "related_commit_url": "none",
        #"cumulative_window_start_sha": "none",
        #"cumulative_window_end_sha": "none",
        #"cumulative_window_commit_count": 0,
        "indirect_recovery_type": "none",
        "confidence": "low",
        "text_evidence_type": "none",
        "indirect_recovery_type": "none",

        # Backward-compatible output
        "related_commit_found": "false",

        # Role-specific columns
        "error_inducing_commit": "",
        "error_inducing_commit_url": "",
        #"error_inducing_cumulative_start_sha": "",
        #"error_inducing_cumulative_end_sha": "",

        "fix_addressing_commit": "",
        "fix_addressing_commit_url": "",
        #"fix_addressing_cumulative_start_sha": "",
        #"fix_addressing_cumulative_end_sha": "",

        # Traceability
        "reason": f"Agent/MCP failed: {error_type}: {error_message}",
        "num_commits_inspected": 0,
        "commits_inspected": "",
        "stop_reason": stop_reason,
        "no_match_reason_type": "",
        
        "agent_raw_output": "",
        "num_mcp_calls": 0,
        "mcp_calls_summary": "",

        # Error details
        "error_type": error_type,
        "error_message": error_message,
        "traceback": traceback.format_exc(),
    }


def build_work_items(df):
    work_items = []

    for pos, row in df.reset_index(drop=True).iterrows():
        window_type = "recovery_window"

        #if not window_type.endswith("_window"):
            #window_type = window_type + "_window"
        
        work_items.append({
            "idx": int(pos),
            "row": row,
            "window": {
                "window_type": window_type,
                "window_start_commit": row.get("last_fail_commit", ""),
                "window_end_commit": row.get("fix_commit", ""),
            },
        })


    return work_items

In [ ]:
from agents.mcp import MCPServerStreamableHttp

TOKEN = TOKEN.strip()

async def test_github_mcp_connection():
    async with MCPServerStreamableHttp(
        name="GitHub MCP",
        params={
            "url": "https://api.githubcopilot.com/mcp/x/repos/readonly",
            "headers": {
                "Authorization": f"Bearer {TOKEN}",
                "X-MCP-Toolsets": "repos",
                "X-MCP-Readonly": "true",
            },
        },
        cache_tools_list=True,
    ) as github_mcp:
        tools = await github_mcp.list_tools()
        print(f"Connected. Tool count: {len(tools)}")
        for tool in tools[:10]:
            print(tool.name)

await test_github_mcp_connection()

In [ ]:
async def main():
    set_default_openai_key(OPENAI_API_KEY)

    outputs = []
    debug_logs = []

    work_items = build_work_items(transitions_with_code_fix_expected)

    for chunk_start in range(0, len(work_items), CHUNK_SIZE):
        chunk = work_items[chunk_start:chunk_start + CHUNK_SIZE]
        chunk_end = chunk_start + len(chunk)

        print(f"Starting MCP chunk {chunk_start + 1}-{chunk_end} / {len(work_items)}")

        try:
            async with MCPServerStreamableHttp(
                name="GitHub MCP",
                params={
                    "url": "https://api.githubcopilot.com/mcp/x/repos/readonly",
                    "headers": {
                        "Authorization": f"Bearer {TOKEN}",
                        "X-MCP-Toolsets": "repos",
                        "X-MCP-Readonly": "true",
                    },
                },
                cache_tools_list=True,
            ) as github_mcp:
                agent = Agent(
                    name="CODE_EXPECTED_WITH_TEST_CONTEXT_SINGLE_COMMIT_WINDOW_SKILL",
                    instructions=CODE_EXPECTED_WITH_TEST_CONTEXT_SINGLE_COMMIT_WINDOW_SKILL,
                    model="gpt-5.6-luna",
                    model_settings=ModelSettings(
                        reasoning={"effort": "medium"},
                        verbosity="low",
                    ),
                    mcp_servers=[github_mcp],
                    mcp_config={
                        "include_server_in_tool_names": True,
                    },
                )

                for item in chunk:
                    idx = item["idx"]
                    row = item["row"]
                    #window = item["window"]

                    print(
                        f"Analyzing source row {idx + 1}/{len(transitions_with_code_fix_expected)}, "
                        f"{row.get('last_fail_commit', '')} -> {row.get('fix_commit', '')} "
                    )

                    prompt = build_row_prompt_commit_window(row)

                    try:
                        result = await Runner.run(
                            agent,
                            prompt,
                            max_turns=20,
                        )

                        raw_output = result.final_output.strip()
                        parsed_output = parse_agent_output_commit_window(raw_output)

                        trace_items, tool_items = extract_trace_and_tool_items(result)

                        result_row = build_commit_window_result_row(
                            row=row,
                            parsed_output=parsed_output,
                        )

                        result_row["source_row_index"] = int(idx)
                        result_row["num_mcp_calls"] = len(tool_items)
                        result_row["mcp_calls_summary"] = " || ".join(
                            item["item_type"] for item in tool_items
                        )
                        result_row["agent_raw_output"] = raw_output
                        result_row["error_type"] = ""
                        result_row["error_message"] = ""
                        result_row["traceback"] = ""

                        outputs.append(result_row)

                        debug_logs.append({
                            "source_row_index": int(idx),
                            "window_type": "recovery_window",
                            "window_start_commit": row.get("last_fail_commit", ""),
                            "window_end_commit": row.get("fix_commit", ""),
                            "prompt": prompt,
                            "raw_output": raw_output,
                            "num_mcp_calls": len(tool_items),
                            "mcp_calls": tool_items,
                            "trace_items": trace_items,
                        })

                        print(
                            parsed_output.get("match_category", ""),
                            parsed_output.get("text_evidence_type", ""),
                            parsed_output.get("indirect_recovery_type", ""),
                            parsed_output.get("confidence", ""),
                            parsed_output.get("no_match_reason_type", ""),
                            parsed_output.get("reason", ""),
                            parsed_output.get("num_commits_inspected", ""),
                            parsed_output.get("stop_reason", ""),
                        )

                    except asyncio.CancelledError as e:
                        error_row = build_error_result_row(
                            row=row,
                            #window=window,
                            idx=idx,
                            error=e,
                            stop_reason="runner_cancelled_error",
                        )
                        outputs.append(error_row)

                        debug_logs.append({
                            "source_row_index": int(idx),
                            "window_type": "recovery_window",
                            "window_start_commit": row.get("last_fail_commit", ""),
                            "window_end_commit": row.get("fix_commit", ""),
                            "prompt": prompt,
                            "error_scope": "runner_run",
                            "error": f"{type(e).__name__}: {e}",
                            "traceback": traceback.format_exc(),
                        })

                        print("Runner cancelled:", type(e).__name__, e)

                    except BaseException as e:
                        error_row = build_error_result_row(
                            row=row,
                            #window=window,
                            idx=idx,
                            error=e,
                            stop_reason="agent_or_mcp_error",
                        )
                        outputs.append(error_row)

                        debug_logs.append({
                            "source_row_index": int(idx),
                            "window_type": "recovery_window",
                            "window_start_commit": row.get("last_fail_commit", ""),
                            "window_end_commit": row.get("fix_commit", ""),
                            "prompt": prompt,
                            "error_scope": "runner_run",
                            "error": f"{type(e).__name__}: {e}",
                            "traceback": traceback.format_exc(),
                        })

                        print("Agent failed:", type(e).__name__, e)

                    save_checkpoint(outputs, debug_logs)

        except asyncio.CancelledError as e:
            debug_logs.append({
                "chunk_start": chunk_start,
                "chunk_end": chunk_end,
                "error_scope": "mcp_chunk_context_cancelled",
                "error": f"{type(e).__name__}: {e}",
                "traceback": traceback.format_exc(),
            })

            print("MCP chunk cancelled during setup/cleanup:", type(e).__name__, e)
            save_checkpoint(outputs, debug_logs)
            continue

        except BaseException as e:
            debug_logs.append({
                "chunk_start": chunk_start,
                "chunk_end": chunk_end,
                "error_scope": "mcp_chunk_context",
                "error": f"{type(e).__name__}: {e}",
                "traceback": traceback.format_exc(),
            })

            print("MCP chunk failed during setup/cleanup:", type(e).__name__, e)
            save_checkpoint(outputs, debug_logs)
            continue

    result_df = pd.DataFrame(outputs)
    result_df.to_csv(RESULTS_FINAL_PATH, index=False)

    with open(DEBUG_FINAL_PATH, "w") as f:
        json.dump(debug_logs, f, indent=2, ensure_ascii=False)

    return result_df, debug_logs

In [ ]:
transitions_with_code_fix_expected_result_df, transitions_with_code_fix_expected_debug_logs = await main()

In [ ]:
transitions_with_code_fix_expected_result_df.to_csv("mcp_agent_code_fix_expected_result_0928_run4.csv")

In [ ]:
transitions_with_code_fix_expected_result_df["match_category"].value_counts()

In [ ]:
run2_code_fix_expected_result_df = pd.read_csv("mcp_agent_code_fix_expected_result_0928_run2.csv")
run2_code_fix_expected_result_df["match_category"].value_counts()

In [ ]:
transitions_with_code_fix_expected_result_df.columns

In [ ]:
transitions_with_code_fix_expected_result_df.head(3)

In [ ]:
run2 = pd.read_csv('mcp_agent_code_fix_expected_result_0928_run2.csv')
run3 = pd.read_csv('mcp_agent_code_fix_expected_result_0928_run3.csv')
run4 = pd.read_csv('mcp_agent_code_fix_expected_result_0928_run4.csv')

audit_key_cols = ['source_fix_commit', 'track_id', 'workflow_run_id', 'job_id']
audit_match_categories = {'strong_patch_match', 'weak_patch_match'}

In [ ]:
def normalize_agent_run(frame, source_run):
    out = frame.copy()
    for col in audit_key_cols:
        out[col] = out[col].astype('string').str.lower().str.strip()
    out['_case_key'] = out[audit_key_cols].agg('||'.join, axis=1)
    out['match_category_norm'] = out['match_category'].astype('string').str.strip()
    out = out[out['match_category_norm'].isin(audit_match_categories)].copy()
    out['source_run'] = source_run
    return out

run2_match_rows = normalize_agent_run(run2, 'run2')
run3_match_rows = normalize_agent_run(run3, 'run3')
run4_match_rows = normalize_agent_run(run4, 'run4')
matched_union_rows = pd.concat([run2_match_rows, run3_match_rows, run4_match_rows], ignore_index=True)
print('merged matched rows:', len(matched_union_rows))
print('unique cases before runner filter:', matched_union_rows['_case_key'].nunique())


In [ ]:
# Join runner metadata before applying the AMD/ROCm filter.
actions_for_join = df_actions.copy() if 'df_actions' in globals() else pd.read_csv('pytorch_actions_0928.csv')
def add_numeric_join_keys(frame):
    out = frame.copy()
    out['workflow_run_id_key'] = pd.to_numeric(out['workflow_run_id'], errors='coerce').astype('Int64').astype('string')
    out['job_id_key'] = pd.to_numeric(out['job_id'], errors='coerce').astype('Int64').astype('string')
    return out
matched_union_k = add_numeric_join_keys(matched_union_rows)
actions_k = add_numeric_join_keys(actions_for_join)
runner_lookup = actions_k[['workflow_run_id_key', 'job_id_key', 'runner_name', 'runner_group_name', 'workflow_event', 'workflow_name', 'job_name', 'completed_at', 'workflow_created_at']].drop_duplicates(['workflow_run_id_key', 'job_id_key'], keep='last')
matched_union_with_runner = matched_union_k.merge(runner_lookup, on=['workflow_run_id_key', 'job_id_key'], how='left', suffixes=('', '_actions'))
rocm_mask = (matched_union_with_runner['runner_name'].astype('string').str.contains('rocm', case=False, na=False) | matched_union_with_runner['runner_group_name'].astype('string').eq('linux.rocm.gpu.group'))
rocm_union = matched_union_with_runner.loc[rocm_mask].copy()
print('ROCm matched rows:', len(rocm_union))
print('ROCm unique cases:', rocm_union['_case_key'].nunique())

In [ ]:
# Widen each ROCm union row with the matched payload from each agent run.
audit_wide = rocm_union.copy()
for source_col, union_col in {'source_run': 'union_source_run', 'transition_id': 'union_transition_id', 'source_fix_commit': 'union_source_fix_commit', 'track_id': 'union_track_id', 'workflow_run_id': 'union_workflow_run_id', 'job_id': 'union_job_id'}.items():
    audit_wide[union_col] = audit_wide[source_col] if source_col in audit_wide else pd.NA

for run_name, run_frame in {'run2': run2_match_rows, 'run3': run3_match_rows, 'run4': run4_match_rows}.items():
    payload_cols = [c for c in run_frame.columns if c not in audit_key_cols + ['_case_key', 'match_category_norm', 'source_run']]
    payload = run_frame[['_case_key'] + payload_cols].drop_duplicates('_case_key', keep='last').rename(columns={c: f'{run_name}_{c}' for c in payload_cols})
    audit_wide = audit_wide.merge(payload, on='_case_key', how='left', validate='many_to_one')
    present = audit_wide['_case_key'].isin(run_frame['_case_key'])
    audit_wide[f'{run_name}_present'] = present
    audit_wide[f'{run_name}_match_status'] = present.map({True: 'matched_case_key', False: 'not_present'})
    for field in ['transition_id', 'match_category', 'confidence', 'num_commits_inspected', 'num_mcp_calls']:
        col = f'{run_name}_{field}'
        if col in audit_wide.columns:
            audit_wide[f'{run_name}_{field}.1'] = audit_wide[col]


In [ ]:
# Failure evidence is keyed by the original transition identity.
evidence_file = 'transitions_with_failure_gemini_0928_run4.csv'
evidence = pd.read_csv(evidence_file)
for col in ['fix_commit', 'track_id', 'last_fail_workflow_run_id', 'last_fail_job_id']:
    evidence[col] = evidence[col].astype('string').str.lower().str.strip()
evidence['_case_key'] = evidence[['fix_commit', 'track_id', 'last_fail_workflow_run_id', 'last_fail_job_id']].agg('||'.join, axis=1)
def join_unique(values):
    result = []
    for value in values:
        if pd.notna(value) and str(value).strip() and str(value) not in result:
            result.append(str(value))
    return ' || '.join(result) if result else pd.NA
evidence_agg = evidence.groupby('_case_key', dropna=False)[['assertion_error_lines', 'failed_tests', 'pytest_tests', 'assertion_category', 'assertion_category_reason']].agg(join_unique).rename(columns={'assertion_error_lines': 'assertion_error_lines_agg', 'failed_tests': 'failed_tests_agg', 'pytest_tests': 'pytest_tests_agg', 'assertion_category': 'assertion_category_agg', 'assertion_category_reason': 'assertion_category_reason_agg'}).reset_index()
audit_wide = audit_wide.merge(evidence_agg, on='_case_key', how='left', validate='many_to_one')
print('evidence matches:', audit_wide['assertion_error_lines_agg'].notna().sum(), '/', len(audit_wide))

In [ ]:
def assertion_cluster(value):
    text = '' if pd.isna(value) else str(value).lower()
    if any(k in text for k in ['tensor-likes are not close', 'tensor-likes are not equal', 'scalars are not equal']): return 'tensor/scalar numerical mismatch'
    if 'clas[' in text or 'new_zeros' in text: return 'generated code/IR textual mismatch'
    if any(k in text for k in ['keys of the mappings do not match', 'keyerror', 'missing key']): return 'mapping/key-set mismatch'
    if any(k in text for k in ['string comparison', 'repr(', 'str(', 'does not match']): return 'string/repr/expected-text mismatch'
    if any(k in text for k in ['shape', 'size', 'dimension', 'symbolic_shapes', 'block_size', 'tma compatibility']): return 'shape/symbolic/tiling constraint failure'
    if 'process group' in text: return 'distributed/process-group invariant failure'
    if any(k in text for k in ['timed out', 'not compiled with cuda', 'libcudart']): return 'timeout/runtime availability failure'
    if any(k in text for k in ['precision flag leak', 'opinfo is broken', 'nested compiled autograd', 'prologue fusion', 'not found in {']): return 'compiler/runtime state invariant failure'
    if any(k in text for k in ['tensor(false', 'false is not true', 'not greater than', 'not less than']): return 'boolean/count predicate mismatch'
    return 'other assertion'



In [ ]:
audit_wide['assertion_error_cluster'] = audit_wide['assertion_error_lines_agg'].map(assertion_cluster)
audit_one_case = audit_wide.drop_duplicates('_case_key', keep='first').reset_index(drop=True)
if 'Unnamed: 0.1' not in audit_one_case.columns:
    audit_one_case.insert(0, 'Unnamed: 0.1', audit_one_case.index + 1)
AUDIT_OUTPUT = 'run2_run3_run4_union_cross_run_agent_audit_with_failure_evidence_one_row_per_case_all_columns.csv'
audit_one_case.to_csv(AUDIT_OUTPUT, index=False)
print('wrote:', AUDIT_OUTPUT)
print('rows:', len(audit_one_case), 'unique cases:', audit_one_case['_case_key'].nunique(), 'columns:', len(audit_one_case.columns))